# RustPower 0.6.0: N-1 and Continuation Power Flow Demo

This notebook demonstrates the new continuation power flow (CPF) workflow and the experimental N-1 screening framework using the PEGASE 9241-bus system. CPU execution works without NVIDIA software. The final section optionally compares the CUDA and hybrid backends.

## 1. Introduction

CPF traces a PV curve by scaling the current net injections and reports whether the directed tangent passes the nose. N-1 screening independently removes each initially active line and two-winding transformer, identifies slackless islands before numerical dispatch, and records convergence and optional security violations.

Run this notebook from `examples/python/`, or adjust `CASE_PATH` below to point to `cases/pegase9241/data.zip`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rustpower as rp

CASE_PATH = Path('../../cases/pegase9241/data.zip').resolve()
CPU_WORKERS = 4

if not CASE_PATH.is_file():
    raise FileNotFoundError(f'PEGASE 9241 case not found: {CASE_PATH}')

print('RustPower:', rp.version())
print('Extension:', rp.__file__)
print('Features:', rp.features())
print('CUDA status:', rp.n1_cuda_status())
print('Case:', CASE_PATH)

## 2. N-1 CPU Demo

The grid uses `qlim=False` by default. N-1 studies keep the PQ/PV/Slack partition fixed. Preparation owns an immutable baseline snapshot; changing the source grid later requires preparing a new study.

In [ ]:
grid = rp.PowerGrid(str(CASE_PATH))
base_report = grid.solve(tol=1e-8, max_iter=20)

print(base_report)
display(grid.res_bus.head())

if not base_report.converged:
    raise RuntimeError('The N-1 baseline power flow did not converge')

In [ ]:
study = grid.prepare_n1()
cpu_report = study.run(workers=CPU_WORKERS, backend='cpu')

print('Faults selected:', len(study.faults))
print(pd.Series(cpu_report.summary))

## 3. Process N-1 Results and Extract Failed Cases

A status-only `run()` aggregates passing scenarios and retains events for rejected numerical scenarios. `outage_kind` and `outage_id` provide stable device identities. `outage_entity` is an internal ECS identifier and is not needed for ordinary analysis.

In [ ]:
events = pd.DataFrame(cpu_report.events)
failed = events[events['outcome'] == 'NumericalFailure']

failed_line_ids = failed.loc[failed['outage_kind'] == 'Line', 'outage_id'].to_numpy(dtype=int)
failed_trafo_ids = failed.loc[failed['outage_kind'] == 'Transformer', 'outage_id'].to_numpy(dtype=int)

print('Numerically failed lines:', failed_line_ids)
print('Numerically failed transformers:', failed_trafo_ids)
display(failed[['outage_kind', 'outage_id', 'iterations', 'residual', 'stage', 'error']])

In [ ]:
islands = pd.DataFrame(cpu_report.islands)

print('Topology-rejected outages:', len(islands))
display(islands[['outage_kind', 'outage_id', 'slackless_bus_count']].head(20))

In [ ]:
if len(cpu_report.numerical_failures) > 0:
    retry_report = study.run(
        cpu_report.numerical_failures,
        initialization='dc',
        workers=CPU_WORKERS,
        backend='cpu',
    )
    print(pd.Series(retry_report.summary))
else:
    print('No numerical failures require a DCPF retry.')

## 4. Continuation Power Flow

CPF solves the base ACPF once, then uses tangent prediction and perpendicular correction from each accepted AC solution. It does not restart every point from DCPF. The default direction scales the present net injections as `S(lambda) = lambda * S_base` with fixed bus types.

In [ ]:
cpf_grid = rp.PowerGrid(str(CASE_PATH))
cpf_result = cpf_grid.run_cpf()

print(cpf_result)
print('Maximum accepted lambda:', cpf_result.lambda_max)
print('Termination:', cpf_result.stop_reason)
print('Statistics:', cpf_result.stats)

if cpf_result.nose is not None:
    print('Estimated nose lambda:', cpf_result.nose.lambda_)
else:
    print('The accepted curve did not confirm passage through the nose.')

In [ ]:
base_voltage = cpf_result.voltages[0]
watched_bus = int(np.abs(base_voltage).argmin())
voltage_magnitude = np.abs(cpf_result.voltages[:, watched_bus])

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(cpf_result.lambda_values, voltage_magnitude, marker='.', label=f'Bus {watched_bus}')

if cpf_result.nose is not None:
    ax.axvline(cpf_result.nose.lambda_, color='tab:red', linestyle='--', label='Estimated nose')

ax.set_xlabel('Loading factor lambda')
ax.set_ylabel('Voltage magnitude (p.u.)')
ax.set_title('PEGASE 9241 continuation power flow')
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

## 5. Experimental CUDA N-1 Power Flow

CUDA and hybrid execution currently use the converged base-AC initialization. The first CUDA call may include driver and device initialization, so a warm-up run is performed before the reported comparison. Scenario classifications must agree across backends; iteration totals may differ because floating-point execution order and linear algebra implementations differ. Correctness is checked independently of performance.

In [ ]:
cuda_available, cuda_diagnostic = rp.n1_cuda_status()
print(cuda_available, cuda_diagnostic)

if cuda_available:
    study.run(backend='cuda')  # Warm up the CUDA driver and device path.
    cuda_report = study.run(backend='cuda', cuda_batch_size=64, cuda_streams=1)
    hybrid_report = study.run(
        workers=CPU_WORKERS,
        backend='hybrid',
        cuda_batch_size=64,
        cuda_streams=1,
    )
else:
    print('Skipping accelerated execution; the existing CPU results remain valid.')

In [ ]:
if cuda_available:
    classification_keys = [
        'processed', 'passed', 'violated', 'numerical_failed', 'islanded',
    ]
    comparison = pd.DataFrame(
        {
            'CPU': cpu_report.summary,
            'CUDA': cuda_report.summary,
            'Hybrid': hybrid_report.summary,
        }
    ).loc[classification_keys]
    display(comparison)
    assert comparison.nunique(axis=1).eq(1).all(), 'Backend scenario classifications differ'

    iteration_counts = pd.Series(
        {
            'CPU': cpu_report.summary['total_iterations'],
            'CUDA': cuda_report.summary['total_iterations'],
            'Hybrid': hybrid_report.summary['total_iterations'],
        },
        name='total_iterations',
    )
    display(iteration_counts)

    timings = pd.DataFrame(
        {
            name: {
                'execution_seconds': result.summary['execution_seconds'],
                'ac_seconds': result.summary['ac_seconds'],
            }
            for name, result in [
                ('CPU', cpu_report),
                ('CUDA', cuda_report),
                ('Hybrid', hybrid_report),
            ]
        }
    ).T
    display(timings)